# SDA-AIE-111 | اليوم الأول
## من صياغة المشكلة إلى معالجة بيانات منافذ (Data Preprocessing)

هذا الدفتر مخصص للطلاب ويعمل على Google Colab. سنستخدم حزمة بيانات **منافذ** المرفقة مع الدورة. نفّذ الخلايا بالترتيب، واكتب إجاباتك في الأماكن المحددة.

**مخرجات اليوم:** صياغة مسألة تعلم آلة، فهم الصفوف والأعمدة، فحص جودة البيانات، معالجة القيم المفقودة، فحص التكرارات والنطاقات والقيم المتطرفة، منع تسرب البيانات، ثم بناء مسار معالجة (Preprocessing Pipeline).

## 1) تجهيز Colab ورفع حزمة البيانات

عند تشغيل الخلية التالية اختر الملف: `SDA-AIE-111_Manafeth_Data_Package (1).zip`. ستفك الخلية الضغط وتحدد مجلد البيانات تلقائيًا.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))

extract_dir = Path('/content/manafeth_course')
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as z:
    z.extractall(extract_dir)

data_dir = extract_dir / 'manafeth_data_package'
print('مجلد البيانات:', data_dir)
print('الملفات المتاحة:')
for path in sorted(data_dir.iterdir()):
    print('-', path.name)

## 2) نشاط صياغة المشكلة

سنحاول تحديد العملاء المتوقع توقفهم عن الطلب خلال الثلاثين يومًا التالية لتاريخ اللقطة. أكمل البطاقة قبل مشاهدة الأعمدة:

- **المستخدم:** ........................................................
- **القرار الذي سيتخذه:** ........................................................
- **المخرج المطلوب:** فئة أم رقم أم مجموعة؟ ........................................................
- **لحظة إصدار النتيجة:** ........................................................
- **كيف نقيس الفائدة؟** ........................................................

لا تكتب اسم خوارزمية؛ المطلوب تعريف المشكلة والقرار.

## 3) قراءة جدول العملاء

نستخدم ملف Parquet لأنه يحافظ على أنواع البيانات أفضل من CSV. كل صف يمثل عميلًا واحدًا في تاريخ لقطة ثابت.

In [ ]:
import pandas as pd
import numpy as np

customers_path = data_dir / 'manafeth_customers.parquet'
df = pd.read_parquet(customers_path)

print('الحجم:', df.shape)
display(df.head())

### تمرين 1: افهم الصف والهدف

1. كم عدد الصفوف والأعمدة؟
2. ماذا يمثل الصف الواحد؟
3. ما معنى العمود `churned_30d`؟
4. لماذا تعد المسألة تصنيفًا ثنائيًا (Binary Classification)؟

In [ ]:
print('أنواع الأعمدة وعدد القيم الموجودة:')
df.info()

print('\nنسبة الفئة المستهدفة:')
print(df['churned_30d'].value_counts(normalize=True).sort_index())

## 4) فحص القيم المفقودة (Missing Values)

القيمة المفقودة لا تعني صفرًا دائمًا. قد يكون غيابها خطأ تسجيل، أو قد يحمل معنى مهمًا.

In [ ]:
missing_count = df.isna().sum()
missing_pct = df.isna().mean().mul(100).round(1)

missing_report = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct,
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

display(missing_report)

### تمرين 2: فسّر الفقد قبل معالجته

- ما نسبة الفقد في `avg_rating`؟
- ما نسبة الفقد في `last_promo_used`؟
- هل غياب `last_promo_used` قد يعني أن العميل لم يستخدم عرضًا؟
- لماذا سيكون تحويل `avg_rating` المفقود إلى صفر قرارًا خاطئًا؟

اكتب قرارك لكل عمود في جملة واحدة.

In [ ]:
clean = df.copy()

# غياب التقييم قد يحمل معلومة، لذلك نحتفظ بمؤشر منفصل
clean['rating_missing'] = clean['avg_rating'].isna().astype('int8')

# مثال تعليمي: التعويض بالوسيط
rating_median = clean['avg_rating'].median()
clean['avg_rating'] = clean['avg_rating'].fillna(rating_median)

# غياب العرض يعني هنا: لم يستخدم عرضًا سابقًا
clean['last_promo_used'] = clean['last_promo_used'].fillna('never_used')

print('القيم المفقودة بعد المعالجة:')
print(clean[['avg_rating', 'last_promo_used']].isna().sum())

> **مهم:** الخلية السابقة توضح الفكرة فقط. عند بناء النموذج لا نحسب الوسيط من كامل البيانات. نقسم البيانات أولًا، ثم يتعلم مسار المعالجة قيمة الوسيط من بيانات التدريب فقط.

## 5) فحص التكرار والنطاقات

In [ ]:
checks = {
    'duplicate_rows': int(df.duplicated().sum()),
    'duplicate_customer_ids': int(df['customer_id'].duplicated().sum()),
    'rating_outside_1_5': int((~df['avg_rating'].dropna().between(1, 5)).sum()),
    'promo_rate_outside_0_1': int((~df['promo_usage_rate'].between(0, 1)).sum()),
    'non_positive_basket': int((df['avg_basket_sar'] <= 0).sum()),
}

display(pd.Series(checks, name='count').to_frame())

### تمرين 3: اتخذ قرار جودة

إذا كانت جميع النتائج صفرًا، هل نتجاوز هذه الفحوص في المرة القادمة؟ اشرح لماذا نجاح الفحص لا يعني أن الفحص غير مهم.

## 6) فحص القيم المتطرفة (Outliers)

القيمة الكبيرة ليست خطأ تلقائيًا. نكتشفها ثم نفهم معناها في العمل قبل الحذف أو التعديل.

In [ ]:
basket = df['avg_basket_sar']
q1, q3 = basket.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr

high_baskets = df.loc[
    basket > upper,
    ['customer_id', 'city', 'avg_basket_sar', 'orders_per_month']
].sort_values('avg_basket_sar', ascending=False)

print('الوسيط:', round(basket.median(), 2))
print('أعلى قيمة:', round(basket.max(), 2))
print('حد IQR:', round(upper, 2))
display(high_baskets.head(10))

### تمرين 4: خطأ أم حالة حقيقية؟

راجع أعلى عشر قيم في `avg_basket_sar`. هل يوجد دليل كافٍ لحذفها؟ اكتب قرارك والسبب.

## 7) كشف تسرب البيانات (Data Leakage)

لحظة التنبؤ هي **1 نوفمبر 2025**. أي معلومة لم تكن متاحة في هذه اللحظة لا يجوز أن تدخل النموذج.

In [ ]:
candidate_columns = [
    'customer_id',
    'city',
    'days_since_last_order',
    'avg_rating',
    'refund_issued',
    'support_ticket_after_snapshot',
    'next_month_orders',
]

pd.DataFrame({'candidate_column': candidate_columns})

### تمرين 5: متاح أم غير متاح؟

صنّف كل عمود في القائمة السابقة إلى:

- متاح وقت التنبؤ ويمكن دراسته كخاصية.
- معرّف لا يصف السلوك.
- هدف نريد للنموذج تعلمه.
- معلومة مستقبلية تسبب تسربًا.

ثم قارن إجابتك مع الخلية التالية.

In [ ]:
target = 'churned_30d'
leak_columns = [
    'refund_issued',
    'support_ticket_after_snapshot',
    'next_month_orders',
]
drop_columns = ['customer_id', target] + leak_columns

X = df.drop(columns=drop_columns)
y = df[target]

print('X:', X.shape)
print('y:', y.shape)
print('أعمدة مستبعدة:', drop_columns)

## 8) تقسيم البيانات قبل تعلم خطوات المعالجة

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('التدريب:', X_train.shape, 'الاختبار:', X_test.shape)
print('نسبة التوقف في التدريب:', round(y_train.mean(), 3))
print('نسبة التوقف في الاختبار:', round(y_test.mean(), 3))

## 9) بناء مسار المعالجة (Preprocessing Pipeline)

سنستخدم مسارًا للأعمدة الرقمية ومسارًا للأعمدة الفئوية. يتعلم كل مسار من بيانات التدريب فقط.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_columns = X_train.select_dtypes(include='number').columns.tolist()
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),
    ('scale', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore')),
])

preprocess = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])

print('أعمدة رقمية:', len(numeric_columns))
print('أعمدة فئوية:', len(categorical_columns))

In [ ]:
X_train_ready = preprocess.fit_transform(X_train)
X_test_ready = preprocess.transform(X_test)

print('شكل التدريب بعد المعالجة:', X_train_ready.shape)
print('شكل الاختبار بعد المعالجة:', X_test_ready.shape)
print('هل بقيت قيم مفقودة؟', bool(np.isnan(X_train_ready.data if hasattr(X_train_ready, 'data') else X_train_ready).any()))

### تمرين 6: اشرح المسار بكلماتك

أكمل الجمل:

1. عالجنا الأعمدة الرقمية باستخدام ................................ و................................
2. عالجنا الأعمدة الفئوية باستخدام ................................ و................................
3. استخدمنا `fit_transform` مع التدريب لأن ................................
4. استخدمنا `transform` فقط مع الاختبار لأن ................................

## 10) التسليم النهائي لليوم الأول

سلّم ملف Colab بعد حفظه باسم المجموعة، ويجب أن يحتوي على:

1. بطاقة صياغة المشكلة مكتملة.
2. تقرير القيم المفقودة مع تفسيرها.
3. نتائج فحص التكرار والنطاقات.
4. قرار التعامل مع القيم المتطرفة.
5. قائمة الأعمدة المستبعدة وأسبابها.
6. مسار معالجة يعمل على بيانات التدريب والاختبار.
7. إجابات تمارين الشرح المكتوبة.

**طريقة الحفظ:** File → Save a copy in Drive، ثم مشاركة الرابط حسب تعليمات الدورة.